# YaSpeech: автопротоколирование деловых встреч на Yandex Cloud
### SpeechKit (ASR) + YandexGPT (многоходовый LLM-анализ) + Object Storage

Кукбук основан на архитектуре реального продакшн-сервиса **YaSpeech** — системы автоматического протоколирования встреч, которую компания «Стройтехэксперт» использует для фиксации решений и задач по строительным проектам. Ниже — упрощённая, но рабочая Python-версия того же пайплайна, которую можно прогнать в Colab/Jupyter.


# 1. Введение

## Назначение

Этот кукбук показывает, как построить сервис **«запись встречи → готовый протокол»** на трёх сервисах Yandex Cloud:

- **SpeechKit (STT v3, async)** — распознавание речи; разделение говорящих берётся из `channelTag`
- **YandexGPT** (через OpenAI-совместимое API) — многоходовый LLM-анализ транскрипта: понимание контекста, коррекция ошибок ASR, определение того, кто есть кто среди говорящих, генерация итогового протокола
- **Object Storage (S3-совместимый)** — хранение аудио, промежуточных транскриптов и готовых протоколов

## Реальный прототип

В проде (Node.js, Yandex Cloud Functions + API Gateway + YMQ) сервис называется **YaSpeech** и решает конкретную бизнес-задачу: прорабы и менеджеры на стройке записывают планёрку на диктофон, а через несколько минут получают структурированный протокол — решения, задачи с ответственными и дедлайнами, открытые вопросы.

## Уникальная фишка, которую разбирает этот кукбук

Большинство ASR-ботов подписывают спикеров как «Спикер 1», «Спикер 2» — и на этом всё. YaSpeech идёт дальше: **диаризация по составу участников проекта**. Сервис заранее знает команду проекта (имена и роли — прораб, заказчик, подрядчик) и передаёт этот список в промпт YandexGPT вместе с транскриптом. Модель ищет в тексте признаки — прямые обращения по имени, характерную лексику роли, кто на какие вопросы отвечает — и сопоставляет реплики с конкретными людьми, а не абстрактными номерами.

## Что вы получите после прохождения кукбука

- Работающий (в пределах Colab) пайплайн: аудио → транскрипт с диаризацией → исправленный текст → протокол в JSON
- Понимание, зачем нужен многоходовый (multi-pass) LLM-анализ, а не один большой промпт
- Готовые промпты, адаптированные из продакшн-кода, которые можно переиспользовать
- Представление о том, как масштабировать это на реальный сервис (очереди, чекпоинты, map-reduce для длинных встреч)

## Сервисы Yandex Cloud, используемые в кукбуке

| Сервис | Назначение |
|---|---|
| SpeechKit STT v3 | Асинхронное распознавание речи; разделение говорящих по `channelTag` |
| YandexGPT (Model Gallery) | Многоходовый анализ: контекст, диаризация, коррекция, идентификация спикеров, генерация протокола |
| Object Storage | Хранение аудио, транскриптов и итоговых протоколов |


# 2. Архитектура решения

## Схема пайплайна

![Архитектура пайплайна YaSpeech](https://raw.githubusercontent.com/Gumbatali/YaSpeech-Public/cookbook/architecture.png)

Пунктирные стрелки — состав команды проекта (имена и роли), который передаётся
в оба LLM-прохода диаризации (A1b) и идентификации спикеров (B2). Это и есть
ключевая архитектурная особенность сервиса — см. раздел 5.6.

## Таблица сервисов

| Сервис | Назначение | В этом кукбуке |
|---|---|---|
| **SpeechKit STT v3** | Распознавание речи, разделение говорящих по `channelTag` | `requests` к `recognizeFileAsync` + `getRecognition` |
| **YandexGPT** | Многоходовый LLM-анализ (5 проходов) | `openai` SDK, OpenAI-совместимый endpoint `gpt://{folder_id}/{model}/latest` |
| **Object Storage** | Хранение аудио и артефактов | `boto3`, S3-совместимый endpoint `storage.yandexcloud.net` |

## Почему диаризация SpeechKit — не последняя инстанция

На реальных записях диаризация по каналу/паузам (`channelTag`) часто ошибается: если оба собеседника говорят в один микрофон, ASR может резать одну непрерывную реплику на "спикеров" через каждые несколько слов. Присваивать имена такой разметке бессмысленно. Поэтому сервис не доверяет диаризации SpeechKit как последней инстанции — заново режет текст на реплики через LLM (проход A1b), опираясь на известный состав участников, и только потом присваивает реальные имена (проход B2).

## Почему не один большой промпт

Продакшн-версия сервиса намеренно разбивает анализ на несколько проходов (multi-pass), а не пытается получить всё одним вызовом:

1. Каждый проход решает одну узкую задачу — легче контролировать качество и стоимость.
2. Ошибка в одном проходе (например, неверно угаданное имя спикера) не портит остальные этапы.
3. Разные проходы требуют разной температуры и разного объёма контекста — коррекция текста должна быть консервативной (`temperature≈0.2-0.25`), а сводка протокола чуть более гибкой.
4. Длинные тексты (диаризация, REFINE) обрабатываются чанками с передачей контекста хвоста предыдущего чанка — иначе модель либо теряет согласованность нумерации спикеров, либо начинает грубо обобщать вместо построчной разметки.


# 3. Подготовка окружения

Устанавливаем зависимости и настраиваем клиентов для трёх сервисов: SpeechKit (через `requests`, т.к. это отдельный REST API), YandexGPT (через OpenAI-совместимый SDK) и Object Storage (через `boto3`).

Понадобятся:
- **FOLDER_ID** — идентификатор каталога: https://yandex.cloud/ru/docs/resource-manager/operations/folder/get-id
- **YC_API_KEY** — API-ключ сервисного аккаунта для вызова YandexGPT и SpeechKit: https://yandex.cloud/ru/docs/iam/operations/api-key/create
- **AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY** — статические ключи для Object Storage: https://yandex.cloud/ru/docs/storage/operations/access-keys/create


In [ ]:
import os
from google.colab import userdata

os.environ["FOLDER_ID"] = userdata.get("FOLDER_ID")
os.environ["YC_API_KEY"] = userdata.get("YC_API_KEY")
os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["BUCKET_NAME"] = "yaspeech-cookbook-demo"
os.environ["GPT_MODEL"] = "yandexgpt-lite"

print("Готово — переменные окружения проставлены.")

In [ ]:
!pip install openai python-dotenv boto3 requests


In [ ]:
import os
import json
import time
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import boto3
import requests
import openai

In [ ]:
# ── Параметры Yandex Cloud ──────────────────────────────────────────────────
FOLDER_ID = os.getenv("FOLDER_ID", "YOUR_FOLDER_ID")
YC_API_KEY = os.getenv("YC_API_KEY", "YOUR_API_KEY")
GPT_MODEL = os.getenv("GPT_MODEL", "yandexgpt-lite")  # Lite выбран по цене/качеству, см. раздел 7
MODEL_URI = f"gpt://{FOLDER_ID}/{GPT_MODEL}/latest"

# ── YandexGPT через OpenAI-совместимое API ──────────────────────────────────
gpt_client = openai.OpenAI(
    api_key=YC_API_KEY,
    base_url="https://llm.api.cloud.yandex.net/v1",
)

# ── Object Storage (S3-совместимый) ─────────────────────────────────────────
S3_BUCKET = os.getenv("BUCKET_NAME", "yaspeech-cookbook-demo")
s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID", "YOUR_AWS_ACCESS_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY", "YOUR_AWS_SECRET_KEY"),
    region_name="ru-central1",
)

# ── SpeechKit STT v3 endpoints ───────────────────────────────────────────────
SPEECHKIT_RECOGNIZE_URL = "https://stt.api.cloud.yandex.net/stt/v3/recognizeFileAsync"
SPEECHKIT_GET_URL = "https://stt.api.cloud.yandex.net/stt/v3/getRecognition"
OPERATION_URL = "https://operation.api.cloud.yandex.net/operations"

print("Клиенты инициализированы:", MODEL_URI)


## 3.1 Проверка подключения к сервисам

Прежде чем запускать полный пайплайн (аудио → протокол), стоит убедиться,
что ключи и каталог настроены правильно — иначе ошибка авторизации всплывёт
где-то в середине многоходового LLM-анализа, потратив время впустую.

Ниже — лёгкая проверка Object Storage (`list_buckets`) и YandexGPT
(минимальный chat-запрос на 5 токенов). SpeechKit использует тот же
`YC_API_KEY`, что и YandexGPT (единый Api-Key сервисного аккаунта) — отдельная
проверка распознавания речи требует реального аудиофайла и выполняется в
разделе 6 при первом реальном запуске пайплайна.


In [ ]:
def check_connections() -> None:
    """Лёгкая проверка окружения перед запуском пайплайна: ловит опечатки в
    ключах/каталоге и проблемы авторизации до того, как они всплывут
    где-то в середине многоходового LLM-анализа."""
    problems = []

    if FOLDER_ID in ("", "YOUR_FOLDER_ID"):
        problems.append("FOLDER_ID не задан (переменная окружения FOLDER_ID)")
    if YC_API_KEY in ("", "YOUR_API_KEY"):
        problems.append("YC_API_KEY не задан (переменная окружения YC_API_KEY)")

    try:
        s3.list_buckets()
        print("Object Storage: подключение работает")
    except Exception as e:
        problems.append(f"Object Storage: {type(e).__name__}: {e}")

    try:
        gpt_client.chat.completions.create(
            model=MODEL_URI,
            messages=[{"role": "user", "content": "ping"}],
            max_tokens=5,
        )
        print("YandexGPT: подключение работает")
    except Exception as e:
        problems.append(f"YandexGPT: {type(e).__name__}: {e}")

    if problems:
        raise RuntimeError("Не настроено окружение:\n- " + "\n- ".join(problems))
    print("Все сервисы доступны, можно запускать пайплайн.")


check_connections()

# 4. Развёртывание инфраструктуры

Создаём бакет Object Storage для хранения аудио и артефактов пайплайна. В реальном сервисе бакет также используется очередью YMQ и Cloud Functions для передачи состояния между асинхронными шагами — здесь мы эмулируем это локальными переменными.


In [ ]:
def ensure_bucket(bucket_name: str) -> None:
    """Создаёт бакет Object Storage, если он ещё не существует."""
    existing = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]
    if bucket_name in existing:
        print(f"Бакет '{bucket_name}' уже существует")
        return
    s3.create_bucket(Bucket=bucket_name)
    print(f"Бакет '{bucket_name}' создан")


ensure_bucket(S3_BUCKET)


In [ ]:
def upload_audio(local_path: str, key: str) -> str:
    """Загружает аудиофайл в Object Storage и возвращает публичный HTTPS URI,
    который SpeechKit сможет прочитать напрямую по ссылке.
    """
    s3.upload_file(local_path, S3_BUCKET, key)
    uri = f"https://storage.yandexcloud.net/{S3_BUCKET}/{key}"
    print(f"Загружено: {uri}")
    return uri


### Конвертация аудио (WAV / M4A / MP3 / OGG → WAV)

SpeechKit STT v3 принимает только контейнеры **WAV**, **OGG (Opus)** и **MP3** — формат M4A (частый экспорт диктофонов, WhatsApp, Telegram) напрямую не поддерживается. Ячейка ниже конвертирует любой входной файл в WAV 16kHz mono через `ffmpeg` (в Colab он уже установлен).

Задайте `AUDIO_INPUT_FORMAT` вручную (`"wav"` или `"m4a"`, также подойдут `"mp3"`/`"ogg"`) — либо оставьте `"auto"`, чтобы формат определился по расширению файла в `AUDIO_INPUT_PATH`.

In [ ]:
import subprocess
from pathlib import Path

AUDIO_INPUT_PATH = "your_meeting_recording.m4a"  # <-- впишите путь к своему аудиофайлу (wav/m4a/mp3/ogg)
AUDIO_INPUT_FORMAT = "auto"  # "auto" | "wav" | "m4a" | "mp3" | "ogg"
SUPPORTED_INPUT_FORMATS = {"wav", "m4a", "mp3", "ogg"}


def resolve_audio_format(path: str, declared_format: str) -> str:
    """Определяет формат аудио: либо явно заданный, либо по расширению файла."""
    if declared_format != "auto":
        fmt = declared_format
    else:
        fmt = Path(path).suffix.lower().lstrip(".")

    if fmt not in SUPPORTED_INPUT_FORMATS:
        raise ValueError(f"Формат '{fmt}' не поддерживается: ожидается один из {SUPPORTED_INPUT_FORMATS}")

    return fmt


def ensure_wav(input_path: str, input_format: str) -> str:
    """Конвертирует аудио в WAV 16kHz mono через ffmpeg. Если уже WAV — ничего не делает."""
    if input_format == "wav":
        print(f"Формат уже WAV: {input_path}")
        return input_path

    output_path = str(Path(input_path).with_suffix(".converted.wav"))
    cmd = ["ffmpeg", "-y", "-i", input_path, "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", output_path]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg не смог сконвертировать {input_format} в wav: {result.stderr[-1000:]}")

    print(f"WAV готов: {output_path}")
    return output_path


resolved_format = resolve_audio_format(AUDIO_INPUT_PATH, AUDIO_INPUT_FORMAT)
AUDIO_WAV_PATH = ensure_wav(AUDIO_INPUT_PATH, resolved_format)
print(f"Итоговый WAV-файл для загрузки в SpeechKit: {AUDIO_WAV_PATH}")

# 5. Логика приложения

Это ядро кукбука. Реализуем упрощённую, но рабочую версию реального пайплайна YaSpeech:

1. **Асинхронное распознавание** через SpeechKit v3 (`recognizeFileAsync` → поллинг операции → `getRecognition`), с разделением по `channelTag`
2. **Постобработка транскрипта**: склейка реплик одного спикера, фильтрация мусорных сегментов
3. **Многоходовый LLM-анализ**:
   - B1 — анализ контекста и домена встречи
   - A1 — диаризация по составу участников (если ASR не смог разделить спикеров)
   - REFINE — построчная коррекция ошибок распознавания по line-ID протоколу
   - B2 — идентификация спикеров **по составу команды проекта** (ключевая фишка YaSpeech)
   - C1 — генерация итогового протокола
4. **Сохранение артефактов** в Object Storage


## 5.1 SpeechKit STT v3: асинхронное распознавание с диаризацией

Реальный сервис запускает распознавание отдельным вызовом (`startRecognition`) и опрашивает операцию отдельными вызовами (`pollRecognitionOnce`), потому что каждый вызов serverless-функции ограничен по времени. Здесь для простоты Colab-примера объединяем это в один блокирующий вызов с циклом ожидания — сама логика API идентична проду.

Ключевые параметры запроса:
- `speakerLabeling.speakerLabeling = "SPEAKER_LABELING_ENABLED"` — включает разметку спикеров на стороне SpeechKit. **Важно:** её результат приходит отдельными событиями, и этот кукбук (как и прод-сервис) их не разбирает — на выходе используется только `channelTag`. Для одноканальной записи он всегда `"0"`, поэтому весь текст схлопывается в одного «Спикера 1» — именно это и чинит LLM-проход A1b ниже
- `textNormalization.literatureText = true` — расставляет пунктуацию и нормализует числа
- Результат приходит НЕ через `operation.response`, а через отдельный endpoint `getRecognition` в формате NDJSON (построчный JSON)


⏱ **Тайминг:** асинхронное распознавание SpeechKit — обычно ~0.3–0.5x от длительности аудио (10-минутная запись → 3-5 минут ожидания, включая поллинг).

In [ ]:
def start_recognition(audio_uri: str, language: str = "ru-RU") -> str:
    """Запускает асинхронное распознавание речи. Возвращает operationId."""
    headers = {"Authorization": f"Api-Key {YC_API_KEY}", "Content-Type": "application/json"}
    body = {
        "uri": audio_uri,
        "recognitionModel": {
            "model": "general",
            "audioFormat": {"containerAudio": {"containerAudioType": "WAV"}},
            "textNormalization": {
                "textNormalization": "TEXT_NORMALIZATION_ENABLED",
                "profanityFilter": False,
                "literatureText": True,
            },
            "languageRestriction": {"restrictionType": "WHITELIST", "languageCode": [language]},
            "audioProcessingType": "FULL_DATA",
        },
        # Диаризация: разделяем реплики по спикерам/каналам
        "speakerLabeling": {"speakerLabeling": "SPEAKER_LABELING_ENABLED"},
    }
    resp = requests.post(SPEECHKIT_RECOGNIZE_URL, headers=headers, json=body, timeout=30)
    resp.raise_for_status()
    operation_id = resp.json()["id"]
    print(f"Операция распознавания запущена: {operation_id}")
    return operation_id


def wait_operation(operation_id: str, poll_interval_s: int = 5, max_wait_s: int = 900) -> None:
    """Ждёт завершения асинхронной операции SpeechKit."""
    headers = {"Authorization": f"Api-Key {YC_API_KEY}"}
    waited = 0
    while waited < max_wait_s:
        resp = requests.get(f"{OPERATION_URL}/{operation_id}", headers=headers, timeout=15)
        resp.raise_for_status()
        op = resp.json()
        if op.get("done"):
            if op.get("error"):
                raise RuntimeError(f"SpeechKit error: {op['error']}")
            return
        time.sleep(poll_interval_s)
        waited += poll_interval_s
    raise TimeoutError("SpeechKit: распознавание не завершилось за отведённое время")


def fetch_recognition(operation_id: str) -> List[Dict[str, Any]]:
    """Забирает результат распознавания (NDJSON) и превращает в список чанков.

    Каждый чанк: { channelTag, text, words } — channelTag это номер канала (0/1/...).
    На одноканальной записи он всегда "0": разделение говорящих даёт LLM-проход A1b.
    """
    headers = {"Authorization": f"Api-Key {YC_API_KEY}"}
    resp = requests.get(
        SPEECHKIT_GET_URL, headers=headers, params={"operationId": operation_id}, timeout=60
    )
    resp.raise_for_status()

    chunks = []
    for line in resp.text.splitlines():
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        final = obj.get("result", {}).get("final")
        if not final:
            continue
        alternatives = final.get("alternatives", [])
        if not alternatives:
            continue
        text = alternatives[0].get("text", "").strip()
        if not text:
            continue
        chunks.append({
            "channelTag": final.get("channelTag", "0"),
            "text": text,
            "words": alternatives[0].get("words", []),
        })
    return chunks


def recognize_meeting_audio(audio_uri: str) -> List[Dict[str, Any]]:
    """Полный синхронный прогон распознавания (для интерактивного Colab-примера;
    в проде старт и поллинг — это разные вызовы serverless-функции)."""
    operation_id = start_recognition(audio_uri)
    wait_operation(operation_id)
    return fetch_recognition(operation_id)

print("✅ 5.1 SpeechKit: функции start_recognition/wait_operation/fetch_recognition готовы")


## 5.2 Постобработка транскрипта

SpeechKit разбивает речь на короткие сегменты. Склеиваем подряд идущие реплики одного спикера, отфильтровываем мусорные обрывки (слова-паразиты, ложные срабатывания на "окей гугл" и т.п.), и приводим к формату `[{speaker, text}]`, удобному для дальнейшего LLM-анализа.


In [ ]:
NOISE_MARKERS = {"угу", "ага", "эм", "эээ", "ну", "вот", "это", "так"}


def is_noise_segment(text: str) -> bool:
    """Мусорный сегмент: короткая реплика, состоящая только из слов-паразитов."""
    words = text.lower().split()
    if not words:
        return True
    if len(words) > 3:
        return False
    return all(w in NOISE_MARKERS for w in words)


def postprocess_transcript(chunks: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    """Склеивает подряд идущие реплики одного спикера и фильтрует мусор."""
    filtered = [c for c in chunks if c["text"].strip() and not is_noise_segment(c["text"])]

    merged: List[Dict[str, str]] = []
    for c in filtered:
        speaker_tag = c["channelTag"]
        if merged and merged[-1]["speaker_tag"] == speaker_tag:
            merged[-1]["text"] += " " + c["text"]
        else:
            merged.append({"speaker_tag": speaker_tag, "text": c["text"]})

    # Присваиваем читаемые метки "Спикер N" по порядку появления
    speaker_ids = sorted({m["speaker_tag"] for m in merged})
    label_map = {tag: f"Спикер {i + 1}" for i, tag in enumerate(speaker_ids)}
    phrases = [{"speaker": label_map[m["speaker_tag"]], "text": m["text"]} for m in merged]
    return phrases


def phrases_to_text(phrases: List[Dict[str, str]]) -> str:
    return "\n".join(f"{p['speaker']}: {p['text']}" for p in phrases)

print("✅ 5.2 Постобработка: функции is_noise_segment/postprocess_transcript готовы")


## 5.3 Вспомогательная функция вызова YandexGPT

Единая обёртка над OpenAI-совместимым API для всех проходов LLM-анализа: system/user промпт → сырой текстовый ответ. Для проходов, ожидающих JSON, добавляем безопасный парсинг с fallback-значением по умолчанию — модель иногда оборачивает JSON в markdown-блок или добавляет лишний текст.


In [ ]:
def call_yandex_gpt(system: str, user: str, temperature: float = 0.2, max_tokens: int = 4000, verbose: bool = False) -> str:
    """Единый вызов YandexGPT через OpenAI-совместимый Chat Completions API.

    verbose=True печатает usage (input/output токены) — удобно, чтобы видеть
    реальную стоимость каждого прохода вместо того, чтобы гадать по факту оплаты.
    """
    response = gpt_client.chat.completions.create(
        model=MODEL_URI,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    if verbose and response.usage:
        u = response.usage
        print(f"    [usage] input={u.prompt_tokens} output={u.completion_tokens} total={u.total_tokens} токенов")

    return response.choices[0].message.content


def parse_json_safely(raw: str, default: Any) -> Any:
    """Достаёт JSON из ответа модели (иногда обёрнут в ```json ... ```)."""
    candidate = raw.strip()
    if candidate.startswith("```"):
        candidate = candidate.split("```")[1]
        candidate = candidate[4:].strip() if candidate.lower().startswith("json") else candidate.strip()
    try:
        return json.loads(candidate)
    except (json.JSONDecodeError, IndexError):
        return default

print("✅ 5.3 YandexGPT-обёртка: call_yandex_gpt/parse_json_safely готовы")


## 5.4 Проход B1: анализ контекста встречи

Определяем тип встречи, предметную область (домен), основные темы и качество распознавания. Это нужно, чтобы последующие промпты (коррекция, идентификация спикеров, протокол) были адресными, а не универсальными — GPT формулирует лучше, зная, что перед ним «строительная планёрка», а не «продажи».

Промпт адаптирован из `promptContextAnalysis` реального сервиса — сохранена ключевая инструкция «не выдумывай, если информации нет — пустой массив/null».


⏱ **Тайминг:** проход B1 — один вызов на весь транскрипт, обычно 3-8 секунд на `yandexgpt-lite`.

In [ ]:
def analyze_context(transcript_text: str, project_name: str) -> Dict[str, Any]:
    system = (
        "Ты — аналитик деловых встреч. Работаешь с транскриптом, полученным через ASR.\n"
        "ЗАДАЧА: понять суть встречи, определить тип, сферу, ключевые сущности.\n"
        "НЕ ВЫДУМЫВАЙ — если информация не упомянута, ставь null или пустой массив.\n"
        "Отвечай только корректным JSON."
    )
    user = f"""Проект: "{project_name}".

Транскрипт:
{transcript_text[:20000]}

Верни JSON:
{{
  "meetingType": "планёрка | статус | переговоры | обучение | прочее",
  "domain": "предметная сфера (например: строительство, IT, финансы, продажи)",
  "mainTopics": ["тема 1", "тема 2", "тема 3"],
  "mentionedEntities": {{
    "people": ["имя"],
    "organizations": ["организация"],
    "dates": ["дата"],
    "amounts": ["сумма или процент"]
  }},
  "transcriptQuality": "good | fair | poor"
}}"""

    raw = call_yandex_gpt(system, user, temperature=0.2, max_tokens=2000)
    default = {
        "meetingType": "прочее", "domain": "не определено", "mainTopics": [],
        "mentionedEntities": {"people": [], "organizations": [], "dates": [], "amounts": []},
        "transcriptQuality": "fair",
    }
    return parse_json_safely(raw, default)

print("✅ 5.4 B1: функция analyze_context готова")


## 5.5 Проход REFINE: коррекция ошибок ASR по line-ID протоколу

Это самая нетривиальная часть пайплайна. ASR систематически искажает специфичную лексику: разрывает слова («гидра изоляция» → «гидроизоляция»), проговаривает числа и марки словами («ка эс два» → «КС-2»). Простой запрос «исправь текст» не годится — модель может выкинуть реплику, перефразировать смысл или незаметно подменить число.

Решение, использованное в проде: **line-ID протокол** вместо JSON. Каждая реплика нумеруется `[N] текст`, и модель обязана вернуть **ровно** тот же набор номеров — это позволяет детерминированно проверить, что ни одна реплика не потерялась (обрыв генерации детектится по недостающим ID, а не тихим fallback'ом на исходный текст).

Дополнительно — **детерминированный валидатор чисел**: сравниваем мультимножества числовых значений до и после коррекции. Если модель поменяла «двадцатого» на «двадцать первое» — это отклоняется программно, а не полагается на «честность» модели.


⏱ **Тайминг:** проход REFINE — по чанкам (25 реплик/чанк), каждый чанк ~5-10 секунд; на часовую встречу (~300-400 реплик, то есть 12-16 чанков) — примерно 1-3 минуты суммарно, плюс retry на чанках с пропущенными id.

In [ ]:
def build_numbered_lines(phrases: List[Dict[str, str]]) -> List[str]:
    """Нумерует реплики: [1] Спикер 1: текст ..."""
    return [f"[{i + 1}] {p['speaker']}: {p['text']}" for i, p in enumerate(phrases)]


def refine_transcript_chunk(numbered_lines: List[str], domain: str) -> str:
    system = f"""Ты — опытный редактор записей деловых переговоров в сфере "{domain}".
Получаешь пронумерованные реплики транскрипта, полученного автоматическим распознаванием речи (ASR).
ASR часто искажает слова: разрывает их, пишет числа и аббревиатуры словами.

ЗАДАЧА: исправить ошибки ASR в каждой реплике, восстановив смысл.

ПРАВИЛА:
1. Читай контекст всего разговора, а не отдельные слова.
2. Склеивай разорванные слова, восстанавливай искажённые термины.
3. Марки, аббревиатуры и обозначения записывай в технической форме ("м триста пятьдесят" -> "М350").
   НИКОГДА не меняй сами числа, даты и суммы.
4. Расставь пунктуацию и заглавные буквы.
5. НЕ добавляй ничего, чего не было сказано. НЕ удаляй и НЕ сокращай сказанное.
6. Формат ответа - СТРОГО построчно, без пояснений, без JSON:
[номер] исправленный текст реплики
7. Сохрани все номера. Каждая входная реплика = ровно одна строка ответа с тем же номером.
8. Каждая входная строка имеет вид "[N] Имя_спикера: текст" — "Имя_спикера:" ЭТО МЕТАДАННЫЕ,
   а не часть реплики. В твоём ответе после "[N] " должен идти ТОЛЬКО исправленный текст самой
   реплики, БЕЗ повторения имени спикера и БЕЗ двоеточия перед текстом."""

    user = "Реплики:\n" + "\n".join(numbered_lines) + "\n\nВерни исправленные реплики построчно в формате [номер] текст."

    return call_yandex_gpt(system, user, temperature=0.25, max_tokens=4000)


def strip_echoed_speaker_prefix(text: str, speaker: str) -> str:
    """Защита от того, что модель повторила 'Спикер:' внутри текста реплики,
    несмотря на прямой запрет в промпте (см. правило 8 в refine_transcript_chunk)."""
    prefix = f"{speaker}:"
    if text.startswith(prefix):
        return text[len(prefix):].strip()
    return text


def parse_refined_lines(raw: str, expected_ids: List[int], phrases: Optional[List[Dict[str, str]]] = None):
    """Парсит ответ модели вида '[N] текст' в map id->текст + список пропущенных id.

    Если передан phrases (исходные реплики), дополнительно вырезает эхо имени
    спикера, если модель всё же продублировала его в начале текста.
    """
    by_id: Dict[int, str] = {}
    for line in raw.splitlines():
        line = line.strip()
        if not line.startswith("["):
            continue
        try:
            idx = line.index("]")
            num = int(line[1:idx])
            text = line[idx + 1:].strip()
        except (ValueError, IndexError):
            continue
        if text:
            if phrases and 1 <= num <= len(phrases):
                text = strip_echoed_speaker_prefix(text, phrases[num - 1]["speaker"])
            by_id[num] = text
    missing = [i for i in expected_ids if i not in by_id]
    return by_id, missing


import re

NUMBER_RE = re.compile(r"\d+")


def extract_numbers(text: str) -> List[int]:
    """Упрощённый валидатор: извлекает только цифровые числа (без разбора числительных
    словами — полная версия описана в system_prompts.md)."""
    return sorted(int(n) for n in NUMBER_RE.findall(text))


def numbers_preserved(original: str, refined: str) -> bool:
    return extract_numbers(original) == extract_numbers(refined)


def refine_transcript(phrases: List[Dict[str, str]], domain: str, chunk_size: int = 25) -> List[Dict[str, str]]:
    """Чанкует транскрипт, исправляет каждый чанк, применяет с валидацией чисел."""
    numbered = build_numbered_lines(phrases)
    result_phrases = [dict(p) for p in phrases]

    for start in range(0, len(numbered), chunk_size):
        chunk = numbered[start:start + chunk_size]
        ids = list(range(start + 1, start + 1 + len(chunk)))

        raw = refine_transcript_chunk(chunk, domain)
        by_id, missing = parse_refined_lines(raw, ids, result_phrases)

        if missing:
            # Один retry на чанк с пропущенными id (обрыв генерации)
            retry_raw = refine_transcript_chunk(chunk, domain)
            retry_by_id, _ = parse_refined_lines(retry_raw, ids, result_phrases)
            for k, v in retry_by_id.items():
                by_id.setdefault(k, v)

        for i in ids:
            original_text = result_phrases[i - 1]["text"]
            candidate = by_id.get(i)
            if candidate is None:
                continue
            if not numbers_preserved(original_text, candidate):
                # Валидатор отклонил правку — оставляем исходный текст
                continue
            result_phrases[i - 1]["text"] = candidate

    return result_phrases

print("✅ 5.5 REFINE: функции build_numbered_lines/refine_transcript готовы")


## 5.6 Проход A1b: диаризация по составу проекта

Это ключевое архитектурное отличие YaSpeech от типового ASR-бота — и оно решает проблему серьёзнее, чем просто "подписать спикеров именами".

Диаризация SpeechKit (`channelTag`/`speakerLabeling`) на реальных записях часто ошибается: если оба собеседника говорят в один микрофон/канал, ASR может резать одну непрерывную реплику на "спикеров" через каждые несколько слов. Присваивать имена такой разметке бессмысленно — сломанной остаётся сама структура реплик, а не только подписи.

Поэтому вместо доверия готовой диаризации ASR сервис заново **режет весь сплошной текст на реплики** через LLM, опираясь на известный состав участников проекта (реальные имена + роли), а не на угадывание количества голосов по звуку. Модель получает список участников как сильную подсказку по числу говорящих и определяет границы реплик по смыслу — вопрос/ответ, обращение по имени, смена темы или роли.

**Важный практический нюанс, найденный экспериментально:** для длинного транскрипта нельзя отдавать весь текст в LLM одним вызовом — модель начинает грубо обобщать (сводит 15-20 минут разговора к нескольким гигантским репликам вместо построчной разметки). Текст режется на чанки по несколько тысяч символов, а хвост предыдущего чанка передаётся как контекст в следующий — иначе нумерация "Спикер 1/2" собьётся на границе чанков.


⏱ **Тайминг:** проход A1b — по чанкам текста (до 4000 символов), каждый чанк ~5-10 секунд; на часовую встречу — обычно 1-3 минуты.

In [ ]:
import re

def chunk_text_for_diarization(text: str, max_chars: int = 4000) -> List[str]:
    """Режет сплошной текст на чанки по границам слов (не разрывая слово пополам).
    Диаризация здесь работает по СПЛОШНОМУ тексту, а не по line-ID протоколу,
    как REFINE — на входе ещё нет надёжной построчной структуры реплик."""
    words = text.split(" ")
    chunks: List[str] = []
    current: List[str] = []
    current_chars = 0
    for word in words:
        if current_chars + len(word) + 1 > max_chars and current:
            chunks.append(" ".join(current))
            current, current_chars = [], 0
        current.append(word)
        current_chars += len(word) + 1
    if current:
        chunks.append(" ".join(current))
    return chunks


def diarize_by_project_team(
    raw_text: str,
    domain: str,
    participants: List[str],
) -> List[Dict[str, str]]:
    """Разбивает сплошной текст на реплики по спикерам, опираясь на состав
    участников встречи. Возвращает список {"speaker": "Спикер N", "text": "..."}.

    Работает по чанкам: одного вызова на длинный текст недостаточно — модель
    начинает грубо обобщать вместо построчной разметки (см. markdown выше).
    """
    participants_block = (
        "УЧАСТНИКИ ВСТРЕЧИ (известный состав):\n" + "\n".join(f"- {p}" for p in participants)
        + f"\n\nОжидаемое число говорящих: {len(participants)}. Это подсказка, а не жёсткое "
        "ограничение — доверяй содержанию текста, если оно расходится с числом."
        if participants else "Состав участников неизвестен — определи число говорящих по тексту."
    )

    system = """Ты — эксперт по анализу деловых переговоров.
Получаешь текст расшифровки (возможно, фрагмент длинной записи). Раздели его на
РЕПЛИКИ ПО ПРЕДЛОЖЕНИЯМ/МЫСЛЯМ и припиши каждую конкретному спикеру.

КАК ОПРЕДЕЛЯТЬ СМЕНУ СПИКЕРА:
1. Вопрос -> ответ: смена происходит между вопросом и ответом
2. Обращение по имени: "Роман, ты..." - значит до этого говорил другой
3. Слова согласия/ответа в начале реплики: "Да", "Хорошо", "Понял" - скорее всего другой спикер
4. Смена темы, роли или точки зрения

ВАЖНО ПРО ГРАНУЛЯРНОСТЬ:
- НЕ объединяй фрагмент в 1-2 гигантские реплики. Ожидается МНОГО отдельных реплик.
- Одна реплика = один непрерывный отрезок речи ОДНОГО человека, обычно 1-4 предложения.
- Если сомневаешься - дроби МЕЛЬЧЕ, а не крупнее.

- НЕ добавляй и НЕ удаляй слова из оригинала, только переносишь в атрибуцию.
- Нумеруй спикеров СТРОГО в формате "Спикер 1", "Спикер 2", ... — НИКОГДА не подставляй
  реальные имена участников в поле "speaker", даже если имя явно прозвучало в тексте
  (имена сопоставляются отдельным проходом identify_speakers, а не здесь). Если дан
  контекст предыдущего фрагмента - сохраняй те же номера для тех же людей.
- Отвечай только JSON: {"segments": [{"speaker": "Спикер 1", "text": "..."}]}"""

    chunks = chunk_text_for_diarization(raw_text)
    all_segments: List[Dict[str, str]] = []
    context_tail: Optional[str] = None

    for chunk in chunks:
        context_block = (
            f"\nКОНЕЦ ПРЕДЫДУЩЕГО ФРАГМЕНТА (уже размечен, для согласования нумерации):\n{context_tail}\n"
            if context_tail else ""
        )
        user = f"""Сфера: "{domain}".

{participants_block}
{context_block}
Текст расшифровки:
{chunk}

Верни JSON в формате {{"segments": [{{"speaker": "Спикер 1", "text": "..."}}]}}."""

        raw = call_yandex_gpt(system, user, temperature=0.2, max_tokens=4000)
        result = parse_json_safely(raw, {"segments": []})
        segments = result.get("segments", [])
        if not segments:
            continue
        all_segments.extend(segments)
        # Хвост чанка — контекст для согласованной нумерации спикеров дальше
        tail = segments[-3:]
        context_tail = "\n".join(f"{s['speaker']}: {s['text']}" for s in tail)

    return normalize_speaker_labels(all_segments)


SPEAKER_LABEL_RE = re.compile(r"^Спикер \d+$")


def normalize_speaker_labels(segments: List[Dict[str, str]]) -> List[Dict[str, str]]:
    """Подстраховка на случай, если модель всё же подставила реальное имя вместо
    'Спикер N' в поле speaker (несмотря на явный запрет в промпте выше) — иначе
    identify_speakers (B2) не находит меток в ожидаемом формате и молча теряет
    спикеров, как это произошло в тестовом прогоне."""
    label_map: Dict[str, str] = {}
    normalized: List[Dict[str, str]] = []
    for seg in segments:
        raw_label = seg.get("speaker", "")
        if SPEAKER_LABEL_RE.match(raw_label):
            canonical = raw_label
        else:
            canonical = label_map.setdefault(raw_label, f"Спикер {len(label_map) + 1}")
        normalized.append({"speaker": canonical, "text": seg["text"]})
    return normalized

print("✅ 5.6 A1b: функции chunk_text_for_diarization/diarize_by_project_team готовы")


## 5.7 Проход B2: идентификация спикеров по составу команды проекта

После того как текст поделён на реплики (шаг выше), отдельный проход присваивает меткам `Спикер N` реальные имена и роли из того же списка участников — модель ищет прямые обращения по имени, характерные роли (прораб говорит о технике и сроках, заказчик — о бюджете и приёмке) и указывает уровень уверенности (`high` / `medium` / `low`).


⏱ **Тайминг:** проход B2 — один вызов на весь транскрипт, 3-8 секунд.

In [ ]:
def identify_speakers(
    transcript_text: str,
    project_team: List[Dict[str, str]],
    context: Dict[str, Any],
) -> List[Dict[str, Any]]:
    """Определяет, кто есть кто среди уже размеченных реплик (см. diarize_by_project_team),
    используя тот же список реальных участников проекта — а не абстрактные 'Спикер 1/2'."""

    team_list = "\n".join(f"- {m['name']} ({m.get('role', 'без роли')})" for m in project_team) or "Список участников не задан"

    system = """Ты - аналитик деловых встреч. Отвечай только корректным JSON.
ЗАДАЧА: по транскрипту определить, кто есть кто среди спикеров.
ПРАВИЛА:
- "high" - имя произнесено прямо (обращение или самопредставление)
- "medium" - косвенные признаки (роль, манера речи, тема которой владеет)
- "low" - предположение без доказательств
- Если уверенность < 50% - guessedName: null, confidence: "low"."""

    user = f"""Контекст встречи:
- Тип: {context.get('meetingType')}
- Сфера: {context.get('domain')}
- Темы: {', '.join(context.get('mainTopics', [])) or '-'}

Участники проекта (для сопоставления):
{team_list}

Транскрипт:
{transcript_text[:18000]}

Верни JSON:
{{
  "speakerDrafts": [
    {{
      "label": "Спикер 1",
      "guessedName": "Иванов Алексей или null",
      "guessedRole": "прораб | заказчик | подрядчик | руководитель | null",
      "confidence": "high | medium | low",
      "reasoning": "краткое обоснование"
    }}
  ]
}}"""

    raw = call_yandex_gpt(system, user, temperature=0.2, max_tokens=2000)
    default = {"speakerDrafts": []}
    return parse_json_safely(raw, default).get("speakerDrafts", [])

print("✅ 5.7 B2: функция identify_speakers готова")


## 5.8 Проход C1: генерация итогового протокола

Финальный проход собирает всё воедино: краткую сводку, список принятых решений, задачи с ответственными и дедлайнами, открытые вопросы. Промпт содержит жёсткие негативные ограничения («НЕ ВЫДУМЫВАЙ — пустой массив лучше, чем выдумка») — это критично для протокола, который потом используется как юридически значимый документ по проекту.


⏱ **Тайминг:** проход C1 — один вызов на весь транскрипт, 5-12 секунд (более длинный ответ, чем у B1/B2).

In [ ]:
def extract_protocol(
    transcript_text: str,
    context: Dict[str, Any],
    speaker_map: str,
    project_name: str,
    meeting_date: str,
) -> Dict[str, Any]:
    system = f"""Ты - профессиональный секретарь деловых встреч. Работаешь в сфере "{context.get('domain')}".
Дата встречи: {meeting_date}.

ПРАВИЛА (строго):
1. "decisions" - ТОЛЬКО реально принятые решения ("решили", "утвердили", "договорились"). Иначе - пустой массив.
2. "actionItems" - ТОЛЬКО конкретные задачи с ответственным. Не назван - "Команда".
3. НЕ ВЫДУМЫВАЙ. Если информация не подтверждается транскриптом - пустой массив или null.
4. "openQuestions" - вопросы, которые ОБСУЖДАЛИ, но НЕ РЕШИЛИ.
5. Отвечай только корректным JSON."""

    user = f"""Проект: "{project_name}"
Сфера: {context.get('domain')} / тип: {context.get('meetingType')}
Темы: {', '.join(context.get('mainTopics', []))}

Спикеры:
{speaker_map}

Транскрипт:
{transcript_text[:22000]}

Верни JSON строго по формату:
{{
  "summary": {{"title": "конкретный заголовок из содержания встречи", "overview": "4-6 предложений деловой прозы"}},
  "participants": ["имя из транскрипта"],
  "decisions": ["решение дословно из транскрипта"],
  "actionItems": [{{"owner": "ответственный", "task": "задача", "deadline": "дата или null"}}],
  "openQuestions": ["вопрос, который обсуждали, но не решили"]
}}"""

    raw = call_yandex_gpt(system, user, temperature=0.15, max_tokens=3000)
    default = {
        "summary": {"title": project_name, "overview": ""},
        "participants": [], "decisions": [], "actionItems": [], "openQuestions": [],
    }
    return parse_json_safely(raw, default)

print("✅ 5.8 C1: функция extract_protocol готова")


## 5.9 Сборка полного пайплайна


In [ ]:
def run_meeting_pipeline(
    audio_local_path: str,
    project_name: str,
    project_team: List[Dict[str, str]],
    meeting_date: str,
) -> Dict[str, Any]:
    meeting_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    audio_key = f"audio/{meeting_id}.wav"

    # 1. Загрузка аудио в Object Storage
    audio_uri = upload_audio(audio_local_path, audio_key)

    # 2. Распознавание речи с диаризацией (SpeechKit v3)
    raw_chunks = recognize_meeting_audio(audio_uri)

    # 3. Постобработка: склейка реплик, фильтрация мусора
    phrases = postprocess_transcript(raw_chunks)
    raw_text = phrases_to_text(phrases)
    print(f"Транскрипт: {len(phrases)} реплик, {len(raw_text)} символов")

    # 4. B1: анализ контекста
    context = analyze_context(raw_text, project_name)
    domain = context.get("domain", "не определено")
    print(f"Контекст: тип={context.get('meetingType')}, домен={domain}")

    # 5. A1b: диаризация ВСЕГДА по составу проекта (не доверяем диаризации SpeechKit —
    # см. markdown раздела 5.6, она часто ошибается на реальных записях)
    participant_names = [m["name"] for m in project_team]
    diarized_segments = diarize_by_project_team(raw_text, domain, participant_names)
    if diarized_segments:
        phrases = [{"speaker": s["speaker"], "text": s["text"]} for s in diarized_segments]
        raw_text = phrases_to_text(phrases)
        print(f"Диаризация по составу проекта: {len(phrases)} реплик")

    # 6. REFINE: коррекция ошибок ASR по line-ID протоколу
    refined_phrases = refine_transcript(phrases, domain)
    refined_text = phrases_to_text(refined_phrases)

    # 7. B2: идентификация спикеров по составу команды проекта
    speaker_drafts = identify_speakers(refined_text, project_team, context)
    speaker_map_lines = []
    for s in speaker_drafts:
        name = s.get("guessedName") or "неизвестен"
        role = s.get("guessedRole")
        role_suffix = f" ({role})" if role else ""
        speaker_map_lines.append(f"- {s['label']} = {name}{role_suffix}")
    speaker_map = "\n".join(speaker_map_lines) or "- нет данных"
    print("Идентификация спикеров:\n" + speaker_map)

    # 8. C1: генерация протокола
    protocol = extract_protocol(refined_text, context, speaker_map, project_name, meeting_date)

    # 9. Сохранение артефактов в Object Storage
    artifacts_prefix = f"meetings/{meeting_id}"
    save_json_to_s3(f"{artifacts_prefix}/transcript.json", {"phrases": refined_phrases})
    save_json_to_s3(f"{artifacts_prefix}/protocol.json", protocol)

    return {
        "meeting_id": meeting_id,
        "context": context,
        "speaker_drafts": speaker_drafts,
        "refined_text": refined_text,
        "protocol": protocol,
    }


def save_json_to_s3(key: str, payload: Dict[str, Any]) -> None:
    s3.put_object(
        Bucket=S3_BUCKET,
        Key=key,
        Body=json.dumps(payload, ensure_ascii=False, indent=2).encode("utf-8"),
        ContentType="application/json",
    )
    print(f"Сохранено: s3://{S3_BUCKET}/{key}")

print("✅ 5.9 Пайплайн: run_meeting_pipeline/save_json_to_s3 готовы")


# 6. Тестирование решения

Пример вызова пайплайна на тестовом аудиофайле планёрки. Замените `audio_local_path` на реальный WAV-файл (16kHz, mono — формат, который ожидает SpeechKit в этой конфигурации) и заполните состав команды проекта, чтобы увидеть, как работает диаризация по именам.


⏱ **Ориентировочное время полного пайплайна** (часовая встреча, ~350 реплик): распознавание ~20-30 мин, A1b ~1-3 мин, REFINE ~2-4 мин, B1/B2/C1 ~15-25 сек суммарно. Итого: доминирует SpeechKit, LLM-часть — единицы минут.

In [ ]:
# Состав команды проекта — именно эти данные передаются в промпты
# диаризации (diarize_by_project_team) и идентификации (identify_speakers)
# вместо гадания "Спикер 1 / Спикер 2"
# <-- впишите реальных участников вашей встречи (имя и роль)
project_team = [
    {"name": "Участник 1", "role": "роль (например, прораб)"},
    {"name": "Участник 2", "role": "роль (например, заказчик)"},
    {"name": "Участник 3", "role": "роль (например, инженер ПТО)"},
]

# result = run_meeting_pipeline(
#     audio_local_path="sample_meeting.wav",
#     project_name="ЖК Садовый — корпус 3",
#     project_team=project_team,
#     meeting_date="2026-07-27",
# )
#
# print(json.dumps(result["protocol"], ensure_ascii=False, indent=2))


### Реальный транскрипт: распознавание вашего аудиофайла

Прогоняем настоящую запись через SpeechKit (загрузка в Object Storage → асинхронное
распознавание → постобработка), а не выдуманный пример. Это реальный сетевой вызов —
займёт время (минуты, в зависимости от длительности аудио) и будет стоить по тарифу
SpeechKit (тарифицируется по длительности аудио; кукбук отправляет одноканальный
WAV — актуальные ставки смотрите в калькуляторе Yandex Cloud).

In [ ]:
audio_uri = upload_audio(AUDIO_WAV_PATH, "test-upload/october-street.wav")
print("=" * 100)
print(f"Аудио загружено в Object Storage: {audio_uri}")
print("=" * 100)

raw_chunks = recognize_meeting_audio(audio_uri)
print(f"\nSpeechKit вернул {len(raw_chunks)} сырых чанков")

demo_phrases = postprocess_transcript(raw_chunks)
print(f"После склейки и фильтрации: {len(demo_phrases)} реплик\n")

demo_text = phrases_to_text(demo_phrases)
print("=" * 100)
print("ТРАНСКРИПТ (реальное распознавание):")
print("=" * 100)
print(demo_text)
print("=" * 100)

demo_context = analyze_context(demo_text, project_name="ЖК Садовый")

print("\nРЕЗУЛЬТАТ B1 (анализ контекста):")
print(json.dumps(demo_context, ensure_ascii=False, indent=2))
print("=" * 100)

_ok37 = bool(demo_context.get("domain")) and demo_context.get("domain") != "не определено"
print(f"{'✅' if _ok37 else '⚠️'} B1 (analyze_context): {'домен и контекст определены' if _ok37 else 'модель не вернула домен — проверьте ключи YandexGPT'}")
print(f"   Тип встречи: {demo_context.get('meetingType')} | Домен: {demo_context.get('domain')} | Качество: {demo_context.get('transcriptQuality')}")


In [ ]:
# A1b: не доверяем разметке спикеров от SpeechKit (см. раздел 5.6) — заново
# режем реальный текст на реплики по составу проекта, опираясь на СПЛОШНОЙ
# текст без speaker-меток SpeechKit (raw_text уже содержит склеенные реплики
# из postprocess_transcript, но без привязки к реальным именам участников)
demo_plain_text = " ".join(p["text"] for p in demo_phrases)

print("=" * 100)
print("ВХОДНОЙ СПЛОШНОЙ ТЕКСТ (реальная запись, без разметки спикеров):")
print("=" * 100)
print(demo_plain_text)
print("=" * 100)

demo_diarized = diarize_by_project_team(
    raw_text=demo_plain_text,
    domain=demo_context.get("domain", "строительство"),
    participants=[m["name"] for m in project_team],
)

print("\nРЕЗУЛЬТАТ A1b (диаризация по составу проекта):")
for seg in demo_diarized:
    print(f"[{seg['speaker']}] {seg['text']}")
print("=" * 100)

_ok38 = len(demo_diarized) > 0 and all(s["speaker"].startswith("Спикер ") for s in demo_diarized)
print(f"{'✅' if _ok38 else '⚠️'} A1b (diarize_by_project_team): "
      f"{len(demo_diarized)} реплик, метки {'все в формате Спикер N' if _ok38 else 'ЕСТЬ ОТКЛОНЕНИЯ от формата Спикер N — проверьте prompt'}")


### Полный прогон LLM-части пайплайна на реальном транскрипте

Продолжаем с результата `demo_diarized` выше (реальная диаризация вашей записи)
и прогоняем оставшиеся три прохода (REFINE → B2 → C1), чтобы увидеть итоговый
протокол целиком на настоящих данных. Это тот же код, что вызывается внутри
`run_meeting_pipeline`, просто по шагам и с промежуточным выводом каждого этапа.

In [ ]:
# REFINE: коррекция ошибок ASR по line-ID протоколу
demo_phrases_for_refine = [{"speaker": s["speaker"], "text": s["text"]} for s in demo_diarized]
demo_refined = refine_transcript(demo_phrases_for_refine, domain=demo_context.get("domain", "строительство"))
demo_refined_text = phrases_to_text(demo_refined)

print("=" * 100)
print("REFINE: исправленный транскрипт")
print("=" * 100)
print(demo_refined_text)

_ok_refine = len(demo_refined) == len(demo_phrases_for_refine) and all(p["text"].strip() for p in demo_refined)
print(f"\n{'✅' if _ok_refine else '⚠️'} REFINE: {len(demo_refined)}/{len(demo_phrases_for_refine)} реплик "
      f"{'исправлены корректно' if _ok_refine else 'ЕСТЬ ПРОБЛЕМА — часть реплик потеряна или пуста'}")

# B2: идентификация спикеров по составу команды проекта
demo_speaker_drafts = identify_speakers(demo_refined_text, project_team, demo_context)
def _clean(value):
    return value if value not in (None, "null", "") else None

demo_speaker_map = "\n".join(
    f"- {s['label']} = {_clean(s.get('guessedName')) or 'неизвестен'}"
    + (f" ({_clean(s.get('guessedRole'))})" if _clean(s.get("guessedRole")) else "")
    for s in demo_speaker_drafts
) or "- нет данных"

print("\n" + "=" * 100)
print("B2: идентификация спикеров")
print("=" * 100)
print(demo_speaker_map)

_identified = sum(1 for s in demo_speaker_drafts if s.get("guessedName") not in (None, "null", ""))
print(f"\n{'✅' if _identified > 0 else '⚠️'} B2: распознано {_identified} из {len(demo_speaker_drafts)} спикеров "
      f"{'(это нормально — модель честно ставит null, если не уверена)' if _identified == 0 else ''}")

# C1: генерация итогового протокола
demo_protocol = extract_protocol(
    transcript_text=demo_refined_text,
    context=demo_context,
    speaker_map=demo_speaker_map,
    project_name="ЖК Садовый — корпус 3",
    meeting_date="2026-07-27",
)

print("\n" + "=" * 100)
print("C1: итоговый протокол")
print("=" * 100)
print(json.dumps(demo_protocol, ensure_ascii=False, indent=2))

_has_title = bool(demo_protocol.get("summary", {}).get("title"))
_has_content = bool(demo_protocol.get("decisions") or demo_protocol.get("actionItems") or demo_protocol.get("openQuestions"))
_ok_c1 = _has_title and _has_content
print(f"\n{'✅' if _ok_c1 else '⚠️'} C1: протокол "
      f"{'сгенерирован с заголовком и содержанием' if _ok_c1 else 'НЕПОЛНЫЙ — проверьте transcript_text и context выше'}")
print(f"   Решений: {len(demo_protocol.get('decisions', []))} | Задач: {len(demo_protocol.get('actionItems', []))} | "
      f"Открытых вопросов: {len(demo_protocol.get('openQuestions', []))}")

print("\n" + "=" * 100)
print("✅ ВЕСЬ ДЕМО-ПРОГОН ПРОШЁЛ УСПЕШНО" if (_ok_refine and _ok_c1) else "⚠️ ЕСТЬ ПРОБЛЕМЫ — см. пометки выше")
print("=" * 100)


# 7. Результаты и выводы

## Что демонстрирует этот кукбук

- Полный цикл «аудио → структурированный протокол» на трёх сервисах Yandex Cloud без единой строчки ручной разметки данных.
- Практический паттерн **многоходового LLM-анализа**: вместо одного промпта на всё — последовательность узких, легко тестируемых проходов (контекст → диаризация по составу проекта → коррекция → идентификация имён → протокол).
- **Line-ID протокол** для коррекции текста вместо JSON — приём, который решает проблему "модель тихо теряет часть реплик или переставляет факты" через явную нумерацию и программную сверку.
- **Диаризацию по составу участников проекта** — не доверяем готовой диаризации SpeechKit (на реальных записях она часто режет одну реплику на "спикеров" через каждые несколько слов), а заново делим сплошной текст на реплики через LLM, зная реальные имена и роли команды. Это отдельный проход ДО коррекции текста и идентификации имён — без него дальнейшие шаги работают поверх уже искажённой структуры.

## Честно о метриках

В продакшн-версии сервиса выбор модели `yandexgpt-lite` для коррекции текста обоснован внутренним бенчмарком (`scripts/experiments/llm-refine-bench`), сравнивающим долю восстановленных слов и полноту соответствия line-ID формату между Lite и Pro моделями на реальных строительных транскриптах. Мы сознательно не переносим сюда конкретные цифры этого бенчмарка — они специфичны для узкого домена (строительная терминология) и конкретной выборки записей, и не будут репрезентативны для произвольного пользователя кукбука.

Если вы адаптируете этот пайплайн под свой домен — рекомендуем прогнать собственный мини-бенчмарк: взять 10-20 реальных фрагментов транскриптов вашей предметной области, вручную аннотировать эталонный текст и померить долю совпадения после REFINE-прохода на нескольких моделях (Lite/Pro), прежде чем фиксировать выбор модели в проде.

## Куда двигаться дальше

- Для длинных встреч (>22 000 символов транскрипта) — реализовать map-reduce: разбить на куски по репликам, обработать каждый C1-проходом отдельно, слить результаты программно + один консолидирующий LLM-вызов.
- Вынести старт распознавания и опрос операции в отдельные serverless-функции (Yandex Cloud Functions + очередь YMQ), чтобы не упираться в таймаут одного вызова на длинных аудиозаписях.
- Добавить QA-проход (faithfulness + completeness check), который проверяет каждый пункт протокола на соответствие транскрипту и подсвечивает то, что модель могла упустить — особенно полезно при низком качестве исходной записи.


# 8. Очистка ресурсов

Удаляем тестовый бакет и все объекты в нём, если вы не планируете использовать их дальше.


In [ ]:
def cleanup_bucket(bucket_name: str) -> None:
    """Удаляет все объекты бакета и сам бакет."""
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name):
        objects = page.get("Contents", [])
        if not objects:
            continue
        s3.delete_objects(
            Bucket=bucket_name,
            Delete={"Objects": [{"Key": o["Key"]} for o in objects]},
        )
    s3.delete_bucket(Bucket=bucket_name)
    print(f"Бакет '{bucket_name}' и все объекты удалены")


# cleanup_bucket(S3_BUCKET)  # раскомментируйте, если хотите очистить ресурсы прямо сейчас

print("✅ 8. cleanup_bucket готова (вызов закомментирован — раскомментируйте, если нужна очистка)")


# 9. Полезные ссылки

## Документация Yandex Cloud

- SpeechKit STT v3 (асинхронное распознавание): https://yandex.cloud/ru/docs/speechkit/stt-v3/
- SpeechKit: диаризация и речевая аналитика: https://yandex.cloud/ru/docs/speechkit/stt/speaker-labeling
- YandexGPT в Model Gallery (раздел AI Studio) — обзор: https://yandex.cloud/ru/docs/ai-studio/quickstart/yandexgpt
- OpenAI-совместимое API для YandexGPT: https://yandex.cloud/ru/docs/foundation-models/concepts/openai-compatibility
- Object Storage (S3-совместимое API): https://yandex.cloud/ru/docs/storage/s3/
- Получение API-ключа: https://yandex.cloud/ru/docs/iam/operations/api-key/create
- Идентификатор каталога (FOLDER_ID): https://yandex.cloud/ru/docs/resource-manager/operations/folder/get-id

## Репозиторий примеров

- Примеры Yandex AI Studio API: https://github.com/yandex-ai-studio/yandex-ai-studio-api-examples

## Дополнительные материалы этого кукбука

- `system_prompts.md` — полные тексты промптов из реального сервиса YaSpeech с пояснениями
- `navigation.md` — оглавление и best practices
- `requirements.txt` — точный список зависимостей
